# 02. Browser Agent 与 Agentic RAG

**Browser Agent** 和 **Agentic RAG** 代表了 Agent 与信息交互的两种进化方向：

- Browser Agent：Agent 像人一样浏览网页（DOM 解析 → 元素定位 → 操作）
- Agentic RAG：Agent 自主决定何时检索、检索什么、何时二次检索

> **检查点**：能说出 Browser Agent 和 API Agent 在信息获取上的本质区别。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# Toy Browser Agent
from html.parser import HTMLParser

class LinkExtractor(HTMLParser):
    def __init__(self):
        super().__init__(); self.links = []; self.texts = []
    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for k, v in attrs:
                if k == "href": self.links.append(v)
    def handle_data(self, data):
        if data.strip(): self.texts.append(data.strip())

def browse_page(html: str) -> dict:
    parser = LinkExtractor()
    parser.feed(html)
    return {"links": parser.links, "text": parser.texts}

sample_page = """<html><body>
  <h1>Welcome to Python Docs</h1>
  <p>Python is a programming language.</p>
  <a href="/tutorial">Tutorial</a>
  <a href="/reference">Reference</a>
  <a href="/download">Download</a>
</body></html>"""

page = browse_page(sample_page)
print(f"Page text: {page['text'][:3]}...")
print(f"Links: {page['links']}")

# Agentic RAG: Agent 决定检索 vs 用自己的知识
class AgenticRAG:
    def __init__(self, knowledge_base: dict, retriever):
        self.kb = knowledge_base; self.retriever = retriever
    
    def should_retrieve(self, query: str, confidence: float) -> bool:
        """Agent 自己决定是否需要检索"""
        if confidence < 0.7:
            return True
        if any(kw in query.lower() for kw in ["latest","recent","current","2025"]):
            return True
        return False
    
    def answer(self, query: str, internal_confidence: float = 0.5) -> str:
        if self.should_retrieve(query, internal_confidence):
            docs = self.retriever(query)
            return f"[RAG] Retrieved {len(docs)} docs. Answer: synthesized from documents."
        return f"[Internal] Answering from knowledge (confidence: {internal_confidence})"

# 演示
class ToyRetriever:
    def __call__(self, query: str) -> list[str]:
        return [f"Doc about {query}"]

rag = AgenticRAG({}, ToyRetriever())

for query, conf in [("What is Python?", 0.9), ("Latest Python version", 0.5)]:
    result = rag.answer(query, conf)
    print(f"Q: '{query}' (conf={conf})")
    print(f"A: {result}\n")

Page text: ['Welcome to Python Docs', 'Python is a programming language.', 'Tutorial']...
Links: ['/tutorial', '/reference', '/download']
Q: 'What is Python?' (conf=0.9)
A: [Internal] Answering from knowledge (confidence: 0.9)

Q: 'Latest Python version' (conf=0.5)
A: [RAG] Retrieved 1 docs. Answer: synthesized from documents.



## Agentic RAG vs 传统 RAG

| | 传统 RAG | Agentic RAG |
|---|----------|-------------|
| 检索触发 | 总是检索 | Agent 自主决定 |
| 检索策略 | 单次 top-K | 可能多轮、多源 |
| 查询优化 | 人工设计 | Agent 可改写查询 |
| 适用场景 | Q&A over docs | 复杂研究任务 |

> **检查点**：能设计一个场景，其中 Agentic RAG 明显优于传统 RAG。

## 真实 API 实验

Agentic RAG：LLM 自主决定「直接答 / 先检索 / 多跳检索」（`toy_browser_agent.py --use-api` 见前文；此处为 RAG 决策）。


In [3]:
# Agentic RAG 决策：模型自己判断是否需要检索
questions = [
    "1+1 等于几？",                       # 应该直接答
    "Mem0 在 LoCoMo 基准上的得分是多少？",  # 应该检索（训练数据可能没有）
]
for q in questions:
    decision = chat_json(client, f"""问题：{q}
你可以：1) 直接回答（已知常识）2) 先检索（需要外部/最新信息）。
返回 JSON: {{"action": "answer|retrieve", "reason": "一句话"}}""")
    print(f"Q: {q}\n  → 决策: {decision.get('action')} — {decision.get('reason')}\n")


Q: 1+1 等于几？
  → 决策: answer — 这是基本算术常识，无需检索。



Q: Mem0 在 LoCoMo 基准上的得分是多少？
  → 决策: retrieve — Mem0在LoCoMo基准上的具体得分不是广泛已知的常识，需要外部最新信息来确认。



## 练习

1. 给 Browser Agent 增加 form-filling 能力：解析 `<form>` 标签，填充 input 字段，点击 submit。
2. 实现 multi-hop RAG：第一次检索结果不满意 → Agent 改写查询 → 第二次检索 → 综合两次结果。
3. 思考 Browser Agent 的可靠性问题：网页结构变化（改版）如何导致 Agent 失败？如何检测和适应？